# 02 · Client distribution (non-IID partitioning)

Builds the federated dataset exactly as training does (deterministic from config + seed) and visualises how heterogeneous the simulated IoT gateways are.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import ppfl  # noqa: F401  (Windows OpenMP import-order guard)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ppfl.visualization.plots import set_style
set_style()
%matplotlib inline

In [ ]:
from ppfl.utils.config import load_config
from ppfl.data.federated_dataset import build_federated_dataset, load_or_prepare
CONFIG = ROOT / "configs" / "fedavg.yaml"
OVERLAYS = []   # e.g. [ROOT / "configs" / "synthetic.yaml"] for the synthetic test data
cfg = load_config(CONFIG, overlays=OVERLAYS)
frame, _ = load_or_prepare(cfg)
ds = build_federated_dataset(cfg, frame)
ds.partition_table

In [ ]:
from IPython.display import Image
from ppfl.visualization.plots import plot_client_distribution
path = plot_client_distribution(ds.partition_table.reset_index(), ROOT / "results" / "notebook_figures",
                                cfg.data.dataset == "synthetic", list(ds.zero_day.unseen_attack_classes))
Image(str(path))

## Zero-day holdout check
Held-out classes must be absent from every client's train/val splits.

In [ ]:
unseen = set(ds.zero_day.unseen_attack_classes)
rows = []
for c in ds.clients:
    rows.append({"client": c.client_id,
                 "benign_train": int((c.train.labels == cfg.data.benign_label).sum()),
                 "known_attack_train": int((c.train.labels != cfg.data.benign_label).sum()),
                 "unseen_in_train_or_val": bool((unseen & set(c.train.labels)) | (unseen & set(c.val.labels))),
                 "zero_day_test_rows": len(c.zero_day),
                 "devices": len(set(c.train.devices))})
pd.DataFrame(rows)

## Effect of α on heterogeneity

In [ ]:
from ppfl.data.partition import heterogeneity_index
results = []
for alpha in [0.05, 0.1, 0.3, 0.5, 1.0, 3.0, 10.0, 100.0]:
    d = build_federated_dataset(load_config(CONFIG, [f"partition.dirichlet_alpha={alpha}"], OVERLAYS), frame)
    sizes = d.partition_table.sum(axis=1)
    results.append({"alpha": alpha, "heterogeneity_js": heterogeneity_index(d.partition_table), "size_cv": sizes.std() / sizes.mean()})
res = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(res["alpha"], res["heterogeneity_js"], marker="o", label="Mean JS distance to global class mix")
ax.plot(res["alpha"], res["size_cv"], marker="s", label="Coefficient of variation of client size")
ax.set_xscale("log"); ax.set(xlabel="Dirichlet α", ylabel="Value", title="Lower α → more heterogeneous clients"); ax.legend()